In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/deeploc.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(28303, 16)
['Unnamed: 0', 'ACC', 'Kingdom', 'Partition', 'Membrane', 'Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion', 'Plastid', 'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome', 'Sequence']


,Unnamed: 0,ACC,Kingdom,Partition,Membrane,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Sequence
0,0,Q28165,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,1,Q86U42,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,2,Q0GA42,Metazoa,3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,3,P82349,Metazoa,1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,4,Q7L5N1,Metazoa,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [2]:
from pathlib import Path

VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWYXBZUO")

LABEL_COLUMNS = [
    "Membrane",
    "Cytoplasm",
    "Nucleus",
    "Extracellular",
    "Cell membrane",
    "Mitochondrion",
    "Plastid",
    "Endoplasmic reticulum",
    "Lysosome/Vacuole",
    "Golgi apparatus",
    "Peroxisome",
]

SEQUENCE_COLUMN = "Sequence"
ID_COLUMN = "ACC"

In [6]:
def is_valid_sequence(seq: str) -> bool:
    if not isinstance(seq, str):
        return False
    seq = seq.strip().upper()
    if len(seq) == 0:
        return False
    return all(char in VALID_AMINO_ACIDS for char in seq)


def load_dataset(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    return df


def clean_sequences(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    before = len(df)

    df = df.dropna(subset=[SEQUENCE_COLUMN])
    df[SEQUENCE_COLUMN] = df[SEQUENCE_COLUMN].astype(str).str.strip().str.upper()
    df = df[df[SEQUENCE_COLUMN].apply(is_valid_sequence)]
    df = df.drop_duplicates(subset=[SEQUENCE_COLUMN])

    print("\nSequence cleaning summary:")
    print(f"Rows before cleaning : {before}")
    print(f"Rows after cleaning  : {len(df)}")

    return df


def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in LABEL_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        df[col] = df[col].clip(0, 1)

    df["label_count"] = df[LABEL_COLUMNS].sum(axis=1)

    print("\nLabel count distribution:")
    print(df["label_count"].value_counts().sort_index())

    return df


def build_final_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    final_df = df[[ID_COLUMN, SEQUENCE_COLUMN] + LABEL_COLUMNS].copy()

    print("\nFinal dataframe shape:")
    print(final_df.shape)

    print("\nSample rows:")
    print(final_df.head())

    return final_df


def save_processed_data(df: pd.DataFrame, output_path: str) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"\nProcessed data saved to: {output_path}")


if __name__ == "__main__":
    input_file = r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\raw\deeploc.csv"
    output_file = r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\processed\deeploc_multilabel.csv"

    df = load_dataset(input_file)
    df = clean_sequences(df)
    df = clean_labels(df)
    final_df = build_final_dataframe(df)
    save_processed_data(final_df, output_file)

Dataset loaded successfully.
Shape: (28303, 16)

Columns:
['Unnamed: 0', 'ACC', 'Kingdom', 'Partition', 'Membrane', 'Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion', 'Plastid', 'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome', 'Sequence']

Sequence cleaning summary:
Rows before cleaning : 28303
Rows after cleaning  : 28303

Label count distribution:
label_count
1    15261
2    10570
3     1871
4      476
5      101
6       24
Name: count, dtype: int64

Final dataframe shape:
(28303, 13)

Sample rows:
      ACC                                           Sequence  Membrane  \
0  Q28165  MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...         0   
1  Q86U42  MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...         0   
2  Q0GA42  MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...         1   
3  P82349  MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...         1   
4  Q7L5N1  MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...         0 

In [10]:
import numpy as np
emb = np.load(r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\processed\embeddings\embeddings.npy")
y = np.load(r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\processed\embeddings\multilabel_targets.npy")
acc = np.load(r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\processed\embeddings\accessions.npy", allow_pickle=True)

print("Embeddings shape:", emb.shape)
print("Targets shape:", y.shape)
print("Accessions shape:", acc.shape)

Embeddings shape: (28303, 480)
Targets shape: (28303, 11)
Accessions shape: (28303,)


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

True
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
import json
with open(r"C:\Users\saita\OneDrive\Desktop\Projects\Protein_Seq\data\processed\embeddings\esm2_t33_650M\label_columns.json") as f:
    labels = json.load(f)
print(len(labels), labels)

2 {'label_columns': ['Membrane', 'Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion', 'Plastid', 'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome'], 'num_labels': 11}
